# Read NCI code–CUI map — the base graph of nci_code_cui_map.dat (scoped)

The pinned NCI EVS file `nci_code_cui_map.dat`, as published, for the codes in scope:

1. pin check;
2. the file as a text table ("|"-separated, no header, no quote handling); the column names from `schema/nci_code_cui_map.yaml`;
3. **the scope** (class annotation `scope`): the codes of the ncit-flat base graph. Codes of that graph without a map row are listed;
4. the graph of the rows in scope by the CSVW table description; structural SHACL;
5. writes `nci-code-cui-map.ttl` and its graph hash;
6. the Excel view `nci_code_cui_map.xlsx`, rebuilt from the graph; text hash equal to the source rows in scope;
7. `Provenance.json`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph, URIRef

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.pins import checked_source_bytes, table_file
from cfa.provenance import flat, provenance, write_provenance
from cfa.read import read_delimited_text, read_xlsx_table
from cfa.rule import (check_module, columns, csvw_metadata, graph_iri, key_columns, load_module, module_source,
                      pct, property_iri, table_to_graph)
from cfa.view import schema_sheet, text_sha256, view_frame, write_view

MODULE = SCHEMA_DIR / 'nci_code_cui_map.yaml'
TABLE = 'nci_code_cui_map'
sv = load_module(MODULE)
check_module(sv)
SOURCE = module_source(sv)
OUT = REPO_ROOT / 'base' / SOURCE
KEYS = key_columns(sv, TABLE)
SCOPE_RULE = sv.get_class(TABLE).annotations['scope'].value
print('scope:', SCOPE_RULE)

## The generated contract is current

In [ ]:
CSVW_PATH = SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json'
SHACL_PATH = SCHEMA_DIR / 'shacl' / f'{SOURCE}.shacl.ttl'
meta = csvw_metadata(sv, TABLE, f'{pct(TABLE)}.csv')
assert json.loads(CSVW_PATH.read_text(encoding='utf-8')) == meta, 'CSVW not current: run checks/00_Schema'
assert SHACL_PATH.read_text(encoding='utf-8') == ShaclGenerator(str(MODULE)).serialize(), 'SHACL not current: run checks/00_Schema'
print('contract current')

## 1–3. Pin check, the file as a text table, the scope

In [ ]:
NAME = table_file(SOURCE, TABLE)
table = read_delimited_text(checked_source_bytes(SOURCE, NAME).decode('utf-8'), '|', columns(sv, TABLE), NAME)
assert not table['code'].duplicated().any(), 'a code occurs more than once'
ncit = Graph().parse(REPO_ROOT / 'base' / 'ncit-flat' / 'ncit-flat.ttl')
flat_codes = {str(o) for o in ncit.objects(None, URIRef(property_iri('ncit-flat', 'Thesaurus', 'code')))}
scoped = table[table['code'].isin(flat_codes)].reset_index(drop=True)
WITHOUT_ROW = sorted(flat_codes - set(table['code']))
print(NAME, '| rows', len(table), '| in scope', len(scoped), '| ncit-flat codes without a map row:', len(WITHOUT_ROW))

## 4–5. The graph, structural SHACL, write

In [ ]:
g = table_to_graph(meta, scoped)
conforms, _, report = pyshacl.validate(g, shacl_graph=Graph().parse(SHACL_PATH))
assert conforms, report
TTL = OUT / f'{SOURCE}.ttl'
write_longturtle(g, TTL, {'cui': f'https://kerfors.github.io/cdisc-for-all/def/{SOURCE}/{pct(TABLE)}#'})
written = Graph().parse(TTL)
GRAPH_HASH = graph_sha256(written)
assert GRAPH_HASH == graph_sha256(g)
print('triples', len(g), '| SHACL conforms |', TTL.name, GRAPH_HASH)

## 6–7. The Excel view, Provenance

In [ ]:
view = view_frame(written, sv, TABLE)
SOURCE_HASH = text_sha256(scoped, KEYS)
VIEW_HASH = text_sha256(view, KEYS)
assert VIEW_HASH == SOURCE_HASH, 'the view differs from the source rows in scope'
prov = provenance(SOURCE, [], [MODULE, CSVW_PATH, SHACL_PATH], graph_iri(SOURCE), len(written),
                  GRAPH_HASH, SOURCE_HASH, VIEW_HASH)
prov['scope'] = {'rule': SCOPE_RULE, 'graphs_read': {'ncit-flat': graph_sha256(ncit)},
                 'ncit_flat_codes_without_row': WITHOUT_ROW}
write_provenance(OUT / 'Provenance.json', prov)
XLSX = OUT / 'nci_code_cui_map.xlsx'
readme = [
    'NCIt code to CUI map (NCI EVS nci_code_cui_map.dat), as published, for the codes of the scoped NCIt graph.',
    'The file has no header: code and CUI named as in the NCI EVS ReadMe ("code <pipe> CUI <pipe>"), the two further fields, which the ReadMe does not describe, by position; field 5 is always empty (each line ends with "|").',
    f'Rebuilt from the graph {TTL.name}: rows sorted by code. Every cell is text.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
write_view(XLSX, readme, {TABLE: view}, schema_sheet(sv, [TABLE]), flat(prov))
assert text_sha256(read_xlsx_table(XLSX, TABLE), KEYS) == SOURCE_HASH
print('text hash, source rows in scope = view:', VIEW_HASH, '|', XLSX.name, 'read back equal')